<div dir="rtl">

# الدرس 25: مشروع: من بيانات حقيقية إلى نموذج

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## الخطوة صفر: صنع ملف البيانات

</div>

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1200
districts = np.array(["Fatih", "Kadikoy", "Besiktas", "Uskudar", "Esenyurt"])
base = {"Fatih": 55, "Kadikoy": 80, "Besiktas": 95, "Uskudar": 65, "Esenyurt": 30}   # أرقام مصنوعة

district = rng.choice(districts, size=n, p=[0.22, 0.2, 0.15, 0.2, 0.23])
area = np.clip(rng.normal(110, 35, n), 40, 260).round()
rooms = np.clip(np.round(area / 35 + rng.normal(0, 0.6, n)), 1, 6).astype(int)
age = rng.integers(0, 51, n).astype(float)
floor = rng.integers(0, 16, n)
elevator = (rng.random(n) < np.clip(0.5 + 0.05 * floor - 0.01 * age, 0.05, 0.98)).astype(int)
metro_km = np.clip(rng.exponential(1.2, n), 0.1, 8).round(2)

per_m2 = np.array([base[d] for d in district])
price = area * per_m2 * (1 - 0.006 * age) * (1 + 0.03 * elevator) - 150 * metro_km
price = (price * rng.lognormal(0, 0.12, n)).round(-1)

df = pd.DataFrame({"district": district, "area_m2": area, "rooms": rooms, "building_age": age,
                   "floor": floor, "has_elevator": elevator, "metro_km": metro_km, "price": price})
df.loc[rng.random(n) < 0.06, "building_age"] = np.nan      # قيم ناقصة
df.loc[rng.random(n) < 0.04, "metro_km"] = np.nan
df.loc[rng.choice(n, size=3, replace=False), "area_m2"] = 0   # أخطاء إدخال
df = pd.concat([df, df.sample(5, random_state=1)], ignore_index=True)   # صفوف مكررة
df.to_csv("istanbul_flats.csv", index=False)
print("saved", df.shape)

<div dir="rtl">

## الخطوة الأولى: التحميل والنظر

</div>

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)          # لكي تظهر كل الأعمدة في الطرفية
pd.set_option("display.max_columns", 20)

df = pd.read_csv("istanbul_flats.csv")
print(df.shape)
print(df.head())

In [ ]:
print(df.dtypes)
print()
print("missing values:")
print(df.isna().sum())

In [ ]:
print(df.describe().round(1))
print()
print(df["district"].value_counts())

<div dir="rtl">

## الخطوة الثانية: التنظيف

</div>

In [ ]:
print("rows with area 0:", (df["area_m2"] <= 0).sum())
print("duplicated rows :", df.duplicated().sum())

df = df[df["area_m2"] > 0].drop_duplicates().reset_index(drop=True)
print("after cleaning  :", df.shape)

<div dir="rtl">

## الخطوة الثالثة: الاستكشاف بالرسوم

</div>

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.hist(df["price"], bins=40, color="C0", alpha=0.8)
a1.set_xlabel("price (thousand TL)"); a1.set_ylabel("count")

for d, part in df.groupby("district"):
    a2.scatter(part["area_m2"], part["price"], s=8, label=d, alpha=0.7)
a2.set_xlabel("area (m2)"); a2.set_ylabel("price")
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
print(df.groupby("district")["price"].median().sort_values())

<div dir="rtl">

## الخطوة الرابعة: التقسيم أولاً

</div>

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="price")
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("train:", X_train.shape, " test:", X_test.shape)

<div dir="rtl">

## الخطوة الخامسة: نموذج مرجعي

</div>

In [ ]:
from sklearn.metrics import mean_absolute_error

baseline_pred = np.full(len(y_test), y_train.median())
print("baseline MAE:", round(mean_absolute_error(y_test, baseline_pred)))

<div dir="rtl">

## الخطوة السادسة: خط المعالجة

</div>

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric = ["area_m2", "rooms", "building_age", "floor", "has_elevator", "metro_km"]
categorical = ["district"]

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

Xt = preprocess.fit_transform(X_train)
print("shape after preprocessing:", Xt.shape)
print("feature names:", list(preprocess.get_feature_names_out()))

<div dir="rtl">

### نقارن ثلاثة نماذج

</div>

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

models = {
    "linear regression": LinearRegression(),
    "random forest":     RandomForestRegressor(n_estimators=300, random_state=0),
    "gradient boosting": HistGradientBoostingRegressor(random_state=0),
}
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)]).fit(X_train, y_train)
    mae = mean_absolute_error(y_test, pipe.predict(X_test))
    fitted[name] = pipe
    print(f"{name:18}  test MAE = {mae:7.1f}")

<div dir="rtl">

### أي الميزات يعتمد عليها النموذج؟

</div>

In [ ]:
from sklearn.inspection import permutation_importance

best = fitted["random forest"]
r = permutation_importance(best, X_test, y_test, n_repeats=5, random_state=0,
                           scoring="neg_mean_absolute_error")
for i in np.argsort(r.importances_mean)[::-1]:
    print(f"{X_test.columns[i]:14} MAE gets worse by {r.importances_mean[i]:7.1f}")

<div dir="rtl">

### نحفظ النموذج، ونستخدمه

</div>

In [ ]:
import joblib

joblib.dump(best, "flat_price_model.joblib")
loaded = joblib.load("flat_price_model.joblib")

new_flat = pd.DataFrame([{
    "district": "Kadikoy", "area_m2": 95, "rooms": 3, "building_age": np.nan,
    "floor": 4, "has_elevator": 1, "metro_km": 0.6,
}])
print("predicted price:", round(loaded.predict(new_flat)[0]), "thousand TL")

<div dir="rtl">

## تمرين: ساعد النموذج الخطي

النموذج الخطي لم يفهم أن السعر يأتي من ضرب المساحة في سعر الحي.
حيلة معروفة: درّبه على لوغاريتم السعر، ثم أرجع التوقع بالأسّ.
لوغاريتم حاصل ضرب يساوي مجموع لوغاريتمات، من الدرس 13. فيصبح الضرب جمعاً، والجمع هو ما يفهمه النموذج الخطي.

1. استخدم أداة سايكت ليرن التي تحوّل الجواب:

`TransformedTargetRegressor(regressor=LinearRegression(), func=np.log, inverse_func=np.exp)`

ضعها مكان النموذج الخطي في خط المعالجة.
2. قارن متوسط الخطأ المطلق مع النموذج الخطي العادي.
3. سؤال فهم: لماذا لا يصح أن نضيف عمود "سعر المتر" كميزة؟

</div>

In [ ]:
from sklearn.compose import TransformedTargetRegressor

# ابنِ خط معالجة جديداً، ودرّبه، واطبع متوسط الخطأ المطلق